# Walk Validation
Thin notebook that reuses `validation_plots.shared_validation` plotting functions.

In [ ]:
from pathlib import Path
import os
import subprocess
import tkinter as tk
from tkinter import filedialog
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from imu_features.config import PipelineConfig
import validation_plots.shared_validation as sv


In [ ]:
DATA_ROOT = Path("Data")
PATIENT_ID = "patient_111"
DATE = "2026-04-30"             # e.g. "2026-03-08" or None => latest
USE_TKINTER_CHOOSER = True      # set True to open folder chooser UI

PLOT_PARAMS = {
    "plot_walk_histogram": True,
    "walk_variance_window_sec": 0.50,
    "walk_compare_fs_hz": 10,
    "walk_compare_min_amp": 0.30,          # fallback only
    "walk_compare_threshold_k": 1.0,
    "walk_compare_min_t_sec": 1.0,
    "walk_compare_step_freq_hz": (0.8, 2.3),
    "window_gate": {
        "window_sec": 1.0,
        "walk_min_amp_threshold": 0.30,    # fallback only
        "walk_threshold_k": 1.0,
        "walk_min_duration_s": 1.0,
    },
}

COLUMN_MAPPING_OVERRIDE = {
    "timestamp": ["Timestamp", "timestamp", "time", "Time"],
    "accel_x": ["Accel_X", "accel_x", "acc_x"],
    "accel_y": ["Accel_Y", "accel_y", "acc_y"],
    "accel_z": ["Accel_Z", "accel_z", "acc_z"],
    "useracc_x": ["UserAccel_X", "useracc_x", "linacc_x", "linearacc_x"],
    "useracc_y": ["UserAccel_Y", "useracc_y", "linacc_y", "linearacc_y"],
    "useracc_z": ["UserAccel_Z", "useracc_z", "linacc_z", "linearacc_z"],
    "gyro_x": ["Gyro_X", "gyro_x", "gyr_x"],
    "gyro_y": ["Gyro_Y", "gyro_y", "gyr_y"],
    "gyro_z": ["Gyro_Z", "gyro_z", "gyr_z"],
}

PIPELINE_CONFIG = PipelineConfig()
PIPELINE_CONFIG.column_candidates = COLUMN_MAPPING_OVERRIDE
print("Walk config ready")


In [ ]:
CHOSEN_DIR = None
LAST_TKINTER_DIR_PATH = Path(".data_last_tkinter_dir.txt")

def _last_tkinter_dir(default_dir=None):
    default_path = Path(default_dir or DATA_ROOT)
    try:
        if LAST_TKINTER_DIR_PATH.exists():
            cached = Path(LAST_TKINTER_DIR_PATH.read_text().strip()).expanduser()
            if cached.exists():
                return cached
    except Exception:
        pass
    return default_path

def _remember_tkinter_dir(selected_dir):
    try:
        if selected_dir is not None:
            LAST_TKINTER_DIR_PATH.write_text(str(Path(selected_dir).resolve()))
    except Exception:
        pass

def _activate_notebook_process_for_dialog():
    if os.name != "posix":
        return
    try:
        subprocess.run(
            [
                "osascript",
                "-e",
                (
                    'tell application "System Events" '
                    f'to set frontmost of first process whose unix id is {os.getpid()} to true'
                ),
            ],
            check=False,
            capture_output=True,
            text=True,
        )
    except Exception:
        pass

def choose_directory_tkinter(initial_dir=None):
    try:
        start_dir = _last_tkinter_dir(initial_dir)
        _activate_notebook_process_for_dialog()
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        root.lift()
        root.focus_force()
        root.update()
        root.update_idletasks()
        try:
            root.eval('tk::PlaceWindow . center')
        except Exception:
            pass
        _activate_notebook_process_for_dialog()
        selected = filedialog.askdirectory(
            parent=root,
            initialdir=str(Path(start_dir).resolve()),
            title="Choose activity date folder",
            mustexist=True,
        )
        root.attributes("-topmost", False)
        root.destroy()
        chosen = Path(selected) if selected else None
        _remember_tkinter_dir(chosen)
        return chosen
    except Exception as exc:
        print(f"tkinter chooser unavailable ({exc}); using configured path.")
        return None

CHOSEN_DIR = choose_directory_tkinter(initial_dir=DATA_ROOT) if USE_TKINTER_CHOOSER else None
print("Chosen directory:", CHOSEN_DIR)


In [ ]:
sv.configure(PIPELINE_CONFIG, PLOT_PARAMS)

resolve_date_dir = sv.resolve_date_dir
discover_activity_files = sv.discover_activity_files
load_activity_file = sv.load_activity_file
plot_walk_validation = sv.plot_walk_validation
plot_turn_validation = sv.plot_turn_validation
plot_transition_validation = sv.plot_transition_validation
plot_turn_symmetry_comparison = sv.plot_turn_symmetry_comparison
plot_turn_pair_xcorr = sv.plot_turn_pair_xcorr
plot_transition_pair_xcorr = sv.plot_transition_pair_xcorr

print("shared_validation helpers imported")


## Base Directory

In [ ]:
sv.configure(PIPELINE_CONFIG, PLOT_PARAMS)

if CHOSEN_DIR is not None:
    BASE_DIR = Path(CHOSEN_DIR)
else:
    BASE_DIR = resolve_date_dir(DATA_ROOT, PATIENT_ID, DATE)
    if DATE is None:
        DATE = BASE_DIR.name

print("Base directory:", BASE_DIR)


In [ ]:
def plot_loaded_sensor(df_loaded, sensor_name, x_col, y_col, z_col, show_x=True, show_y=True, show_z=True, show_resultant=True, source_label=None):
    if df_loaded is None:
        print(f"[SKIP] {sensor_name}: no loaded activity dataframe")
        return
    if 'time_s' not in df_loaded.columns:
        print(f"[SKIP] {sensor_name}: loaded dataframe missing time_s")
        return
    needed = [x_col, y_col, z_col]
    missing = [c for c in needed if c not in df_loaded.columns]
    if missing:
        print(f"[SKIP] {sensor_name}: missing columns {missing}")
        return

    t_plot = pd.to_numeric(df_loaded['time_s'], errors='coerce').to_numpy(dtype=float)
    x = pd.to_numeric(df_loaded[x_col], errors='coerce').to_numpy(dtype=float)
    y = pd.to_numeric(df_loaded[y_col], errors='coerce').to_numpy(dtype=float)
    z = pd.to_numeric(df_loaded[z_col], errors='coerce').to_numpy(dtype=float)
    resultant = np.sqrt(x**2 + y**2 + z**2)

    plt.figure(figsize=(10, 4))
    if show_x:
        plt.plot(t_plot, x, label='x', linewidth=1.0)
    if show_y:
        plt.plot(t_plot, y, label='y', linewidth=1.0)
    if show_z:
        plt.plot(t_plot, z, label='z', linewidth=1.0)
    if show_resultant:
        plt.plot(t_plot, resultant, label='resultant', linewidth=1.8, color='black')
    title_suffix = f" ({source_label})" if source_label else ''
    plt.title(f"{sensor_name}{title_suffix}")
    plt.xlabel('Time (s)')
    plt.ylabel('Value')
    plt.grid(alpha=0.3)
    plt.legend(loc='upper right', fontsize=7, framealpha=0.80, borderpad=0.25, labelspacing=0.25, handlelength=1.6)
    plt.show()


## Load Walk File

In [ ]:
sv.configure(PIPELINE_CONFIG, PLOT_PARAMS)
activity_files = discover_activity_files(BASE_DIR, ["walk"])
print("Discovered activity files:")
for k, v in activity_files.items():
    print(f"  {k}: {v}")
WALK_PATH = activity_files.get("walk")
WALK_DF = None
WALK_META = None
if WALK_PATH is not None:
    print(f"\n[PROCESS] walk: {WALK_PATH}")
    WALK_DF, WALK_META = load_activity_file(WALK_PATH)
    print("Loaded plot file:", WALK_PATH)
else:
    print("[SKIP] walk: not available")


## Accelerometer

In [ ]:
SHOW_ACCEL_X = False
SHOW_ACCEL_Y = True
SHOW_ACCEL_Z = False
SHOW_ACCEL_RESULTANT = True
plot_loaded_sensor(WALK_DF, "Accelerometer", "accel_x", "accel_y", "accel_z", SHOW_ACCEL_X, SHOW_ACCEL_Y, SHOW_ACCEL_Z, SHOW_ACCEL_RESULTANT, source_label=WALK_PATH.name if WALK_PATH else None)


## Gyroscope

In [ ]:
SHOW_GYRO_X = False
SHOW_GYRO_Y = False
SHOW_GYRO_Z = True
SHOW_GYRO_RESULTANT = True
plot_loaded_sensor(WALK_DF, "Gyroscope", "gyro_x", "gyro_y", "gyro_z", SHOW_GYRO_X, SHOW_GYRO_Y, SHOW_GYRO_Z, SHOW_GYRO_RESULTANT, source_label=WALK_PATH.name if WALK_PATH else None)


## User Accelerometer

In [ ]:
SHOW_USERACC_X = False
SHOW_USERACC_Y = True
SHOW_USERACC_Z = False
SHOW_USERACC_RESULTANT = True
plot_loaded_sensor(WALK_DF, "User Accelerometer", "useracc_x", "useracc_y", "useracc_z", SHOW_USERACC_X, SHOW_USERACC_Y, SHOW_USERACC_Z, SHOW_USERACC_RESULTANT, source_label=WALK_PATH.name if WALK_PATH else None)


## Walk Feature Plots

In [ ]:
walk_summaries = []
if WALK_DF is not None and WALK_META is not None:
    summary = plot_walk_validation("walk", WALK_DF, WALK_META)
    walk_summaries.append(summary)
    print(summary)
else:
    print("[SKIP] walk feature plots: walk file not loaded")
print("=== Validation Summary ===")
display(pd.DataFrame(walk_summaries).round(3))


In [ ]:
# Calculated linear and nonlinear walk features from the loaded synchronized walk file
from imu_features.walk_features import (
    WALK_NONLINEAR_FEATURE_NAMES,
    extract_walk_features,
    extract_walk_nonlinear_features,
)


sv.configure(PIPELINE_CONFIG, PLOT_PARAMS)
sv._apply_notebook_window_gate_settings()

walk_feature_specs = [
    ("walk_duration", "Linear"),
    ("walking_speed", "Linear"),
    ("walk_acc_mag_mean", "Linear"),
    ("walk_acc_mag_std", "Linear"),
    ("walk_acc_mag_rms", "Linear"),
    ("walk_gyro_mag_std", "Linear"),
    ("walk_jerk_mean", "Linear"),
    ("walk_jerk_std", "Linear"),
    ("step_count", "Linear"),
    ("cadence", "Linear"),
    ("mean_step_time", "Linear"),
    ("step_time_std", "Linear"),
    ("step_time_cv", "Linear"),
    ("step_regularity", "Linear"),
    ("stride_regularity", "Linear"),
    ("walk_dominant_frequency", "Linear"),
    ("walk_spectral_entropy", "Nonlinear"),
]
walk_feature_specs.extend((feature, "Nonlinear") for feature in WALK_NONLINEAR_FEATURE_NAMES)

if WALK_DF is not None and WALK_META is not None:
    walk_features = extract_walk_features(WALK_DF, WALK_META, PIPELINE_CONFIG)
    walk_features.update(extract_walk_nonlinear_features(WALK_DF, WALK_META, PIPELINE_CONFIG))

    # Keep the table aligned with the plotted validation summary from the same selected segment.
    if walk_summaries:
        walk_summary = walk_summaries[0]
        walk_features["walk_duration"] = walk_summary.get("duration_s", np.nan)
        walk_features["step_count"] = walk_summary.get("step_count", np.nan)
        walk_features["cadence"] = walk_summary.get("cadence", np.nan)
        duration = walk_features.get("walk_duration", np.nan)
        walk_features["walking_speed"] = (
            PIPELINE_CONFIG.walk_distance_m / duration
            if np.isfinite(duration) and duration > 0
            else np.nan
        )

    walk_feature_table = pd.DataFrame([
        {
            "Feature": feature,
            "Type": feature_type,
            "Value": walk_features.get(feature, np.nan),
        }
        for feature, feature_type in walk_feature_specs
    ])
    walk_feature_table["Value"] = pd.to_numeric(walk_feature_table["Value"], errors="coerce")
    walk_display_table = walk_feature_table.copy()
    walk_display_table["Value"] = walk_display_table["Value"].map(lambda value: f"{value:.3f}" if pd.notna(value) else np.nan)
    display(walk_display_table)
else:
    print("[SKIP] walk feature table: walk file not loaded")
